# ROBUST04 Phase 2: Adding Contriever Dense Retrieval (4-Way RRF)

## Based on: ROBUST04_Phase1_Fixed_Faithful.ipynb

## Phase 1 Optimized Parameters (MAP 0.2855):
- BM25: k1=0.7, b=0.6
- RM3: fb_docs=15, fb_terms=100, original_query_weight=0.20
- k_rrf=10
- rerank_depth=1000

## Phase 2 Addition:
- Adding **Contriever** as 4th retrieval signal (dense retrieval)
- 4-Way Pure RRF: BM25 + RM3 + MonoT5 + Contriever

## Expected Performance:
- Phase 1: MAP 0.2855
- Phase 2: MAP 0.33-0.36 (target: 0.35)

In [1]:
hugging_face_token = "[REDACTED_HF_TOKEN]"

In [2]:
from huggingface_hub import login
login(hugging_face_token)

In [3]:
# ============================================================================
# JAVA 21 SETUP
# ============================================================================

import os
import subprocess

print("Checking Java version...")

try:
    result = subprocess.run(['java', '-version'], capture_output=True, text=True)
    current_version = result.stderr
    print(f"Current Java:\n{current_version}")
except:
    print("Java not found")

print("\n Installing Java 21...")
!sudo apt-get update -qq
!sudo apt-get install -y openjdk-21-jdk-headless -qq

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["PATH"] = f"{os.environ['JAVA_HOME']}/bin:" + os.environ["PATH"]

print("\n Verifying Java 21 installation...")
!java -version

print("\n" + "="*70)
print("Java 21 ready!")
print("="*70)

Checking Java version...
Current Java:
openjdk version "17.0.17" 2025-10-21
OpenJDK Runtime Environment (build 17.0.17+10-Ubuntu-122.04)
OpenJDK 64-Bit Server VM (build 17.0.17+10-Ubuntu-122.04, mixed mode, sharing)


 Installing Java 21...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 2.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package openjdk-21-jre-headless:amd64.
(Reading database ... 121689 files and directories

In [4]:
# ============================================================================
# INSTALL PACKAGES
# ============================================================================

print("Installing packages...")

!pip install -q transformers>=4.46.0
# Use pyserini 0.22.1 (stable, works with our setup)
!pip install -q pyserini==0.22.1
!pip install -q ir_measures torch torchvision torchaudio sentence-transformers numpy
!pip install faiss-cpu --no-cache

print("All packages installed!")

Installing packages...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 MB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 67.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 132.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 91.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.6/293.6 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.8/304.8 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 161.5 MB/s eta 0:00:00
All packages installed!


In [5]:
import os
import torch
import numpy as np
import transformers
import pyserini
from collections import defaultdict
from pyserini.search.lucene import LuceneSearcher
from pyserini.index.lucene import IndexReader
from pyserini.search.faiss import FaissSearcher, AutoQueryEncoder
import ir_measures
from ir_measures import MAP, nDCG, P
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import warnings
import time

warnings.filterwarnings('ignore', message='Some weights of.*were not initialized')
warnings.filterwarnings('ignore', message='Some parameters are on the meta device')
warnings.filterwarnings('ignore', message='.*overflowing tokens are not returned.*')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
GPU: Tesla T4


In [6]:
# ============================================================================
# GOOGLE DRIVE SETUP
# ============================================================================
from google.colab import drive, userdata
import os
import sys

try:
    IN_COLAB = True
    print("Running in Google Colab")
except:
    IN_COLAB = False
    print("Running locally")

if IN_COLAB:
    print("\nMounting Google Drive...")
    drive.mount('/content/drive')
    print("Google Drive mounted")

    BASE_DIR = '/content/drive/MyDrive/TEXT/FinalProject'

    if os.path.exists(BASE_DIR):
        print(f"Found directory: {BASE_DIR}")
        os.chdir(BASE_DIR)
        print(f"Changed to: {os.getcwd()}")
    else:
        search_paths = [
            '/content/drive/MyDrive/TEXT/FinalProject',
            '/content/drive/MyDrive/FinalProject',
            '/content/drive/MyDrive/TEST/FinalProject',
            '/content/drive/MyDrive',
        ]

        for path in search_paths:
            query_file = os.path.join(path, 'queriesROBUST.txt')
            if os.path.exists(query_file):
                os.chdir(path)
                BASE_DIR = path
                break
else:
    BASE_DIR = os.getcwd()

print(f"\nChecking for required files...")
print(f"Current directory: {os.getcwd()}")

if os.path.exists('queriesROBUST.txt'):
    print(f"  Found: queriesROBUST.txt")
if os.path.exists('qrels_50_Queries'):
    print(f"  Found: qrels_50_Queries")

print("\n" + "="*70)
print("Setup complete!")
print("="*70)

Running in Google Colab

Mounting Google Drive...
Mounted at /content/drive
Google Drive mounted
Found directory: /content/drive/MyDrive/TEXT/FinalProject
Changed to: /content/drive/MyDrive/TEXT/FinalProject

Checking for required files...
Current directory: /content/drive/MyDrive/TEXT/FinalProject
  Found: queriesROBUST.txt
  Found: qrels_50_Queries

Setup complete!


## 1. Configuration

In [7]:
# Model configuration
USE_MONOT5 = True
USE_CONTRIEVER = True  # NEW in Phase 2! (Dense retrieval)

print(f"\n Phase 2 Configuration:")
print(f"  MonoT5 Reranker: {USE_MONOT5}")
print(f"  Contriever Dense Retrieval: {USE_CONTRIEVER} (NEW!)")
print(f"  4-Way Pure RRF (all weights = 1.0)")
print(f"  k_rrf: 10")
print(f"  rerank_depth: 1000")
print(f"\n Phase 1 baseline: MAP 0.2855")
print(f" Phase 2 target: MAP 0.33-0.36")


 Phase 2 Configuration:
  MonoT5 Reranker: True
  Contriever Dense Retrieval: True (NEW!)
  4-Way Pure RRF (all weights = 1.0)
  k_rrf: 10
  rerank_depth: 1000

 Phase 1 baseline: MAP 0.2855
 Phase 2 target: MAP 0.33-0.36


## 2. Load Index and Configure Searchers

In [8]:
print("Loading ROBUST04 index...")
index_reader = IndexReader.from_prebuilt_index('robust04')

# BM25 searcher with OPTIMIZED parameters from Phase 1 tuning
bm25_searcher = LuceneSearcher.from_prebuilt_index('robust04')
bm25_searcher.set_bm25(k1=0.7, b=0.6)  # Optimized!

# RM3 searcher with OPTIMIZED parameters from Phase 1 tuning
rm3_searcher = LuceneSearcher.from_prebuilt_index('robust04')
rm3_searcher.set_bm25(k1=0.7, b=0.6)  # Optimized!
rm3_searcher.set_rm3(fb_docs=15, fb_terms=100, original_query_weight=0.20)  # Optimized!

searchers = [bm25_searcher, rm3_searcher]

print("Configured BM25 + RM3 (Phase 1 optimized)")
print("  BM25: k1=0.7, b=0.6")
print("  RM3: fb_docs=15, fb_terms=100, original_query_weight=0.20")

Loading ROBUST04 index...


lucene-index.robust04.20221005.252b5e.tar.gz: 1.68GB [00:44, 40.8MB/s]                            


Configured BM25 + RM3 (Phase 1 optimized)
  BM25: k1=0.7, b=0.6
  RM3: fb_docs=15, fb_terms=100, original_query_weight=0.20


## 2b. Load Contriever Dense Retrieval (NEW in Phase 2!)

In [9]:
# Dense Retrieval using Sentence-Transformers (Alternative Approach)
# Instead of pre-built Faiss index, we'll use a dense encoder directly

from sentence_transformers import SentenceTransformer
import numpy as np

dense_encoder = None
doc_embeddings_cache = {}

if USE_CONTRIEVER:
    print("\nLoading Dense Retrieval (Sentence-Transformers approach)...")

    try:
        # Use a lightweight but effective model
        # Options: 'sentence-transformers/all-MiniLM-L6-v2' (fast)
        #          'sentence-transformers/all-mpnet-base-v2' (better quality)
        dense_encoder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
        dense_encoder = dense_encoder.to(device)
        print(f"  ✓ Loaded all-MiniLM-L6-v2")

        # Test encoding
        test_emb = dense_encoder.encode("test query", convert_to_tensor=True)
        print(f"  ✓ Test encoding successful (dim={test_emb.shape[0]})")

    except Exception as e:
        print(f"  Dense encoder load failed: {e}")
        USE_CONTRIEVER = False
        dense_encoder = None

def get_dense_scores(query_text, doc_ids, top_k=1000):
    '''
    Compute dense similarity scores for documents.
    Uses caching to avoid re-encoding documents.
    '''
    if dense_encoder is None:
        return []

    try:
        # Encode query
        query_emb = dense_encoder.encode(query_text, convert_to_tensor=True, device=device)

        # Encode documents (with caching)
        doc_texts = []
        valid_ids = []
        for did in doc_ids:
            try:
                if did in doc_embeddings_cache:
                    continue  # Already cached
                doc = index_reader.doc(did)
                if doc:
                    raw = doc.raw()
                    if raw:
                        # Truncate for efficiency
                        doc_texts.append(raw[:500])
                        valid_ids.append(did)
            except:
                pass

        # Encode new documents
        if doc_texts:
            new_embeddings = dense_encoder.encode(doc_texts, convert_to_tensor=True, device=device, batch_size=32)
            for did, emb in zip(valid_ids, new_embeddings):
                doc_embeddings_cache[did] = emb

        # Compute similarities for all requested docs
        scores = []
        for did in doc_ids:
            if did in doc_embeddings_cache:
                sim = torch.nn.functional.cosine_similarity(
                    query_emb.unsqueeze(0),
                    doc_embeddings_cache[did].unsqueeze(0)
                ).item()
                scores.append((did, sim))

        # Sort by score descending
        scores.sort(key=lambda x: x[1], reverse=True)
        return scores[:top_k]

    except Exception as e:
        print(f"  Dense scoring error: {e}")
        return []

print("\n" + "="*70)
print(f"Dense Retrieval: {'Enabled' if dense_encoder else 'Disabled'}")
print("="*70)



Loading Dense Retrieval (Sentence-Transformers approach)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

  ✓ Loaded all-MiniLM-L6-v2
  ✓ Test encoding successful (dim=384)

Dense Retrieval: Enabled


## 3. Load Queries

In [10]:
def load_queries(query_file):
    queries = {}
    with open(query_file, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split(None, 1)
            if len(parts) == 2:
                queries[parts[0]] = parts[1]
    return queries

def load_qrels(qrel_file):
    qrels = defaultdict(dict)
    with open(qrel_file, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 4:
                qrels[parts[0]][parts[2]] = int(parts[3])
    return qrels

queries = load_queries('queriesROBUST.txt')
qrels = load_qrels('qrels_50_Queries')
train_qids = sorted(qrels.keys())
test_qids = sorted([q for q in queries if q not in qrels])

print(f"Loaded {len(queries)} queries ({len(train_qids)} train, {len(test_qids)} test)")

Loaded 249 queries (50 train, 199 test)


## 4. Helper Functions

In [11]:
def classify_query(query_text):
    wc = len(query_text.split())
    return 'short' if wc <= 3 else 'medium' if wc <= 6 else 'long'

## 5. Multi-Method Retrieval (with Contriever)

In [12]:
def multi_method_fusion(query_text, k=1000):
    '''
    Get results from BM25, RM3, and optionally Dense Retrieval.
    '''
    # BM25
    bm25_hits = searchers[0].search(query_text, k=k)
    bm25_results = [(h.docid, h.score) for h in bm25_hits]

    # RM3
    rm3_hits = searchers[1].search(query_text, k=k)
    rm3_results = [(h.docid, h.score) for h in rm3_hits]

    # Dense Retrieval (using cached embeddings)
    dense_results = []
    if USE_CONTRIEVER and dense_encoder is not None:
        # Get documents from BM25 to score with dense model
        # (We can't search the full corpus, so we re-rank BM25 results)
        bm25_docids = [docid for docid, _ in bm25_results[:k]]
        dense_results = get_dense_scores(query_text, bm25_docids, top_k=k)

    return bm25_results, rm3_results, dense_results


## 6. Load MonoT5 Reranker

In [13]:
monot5_model, monot5_tokenizer = None, None
if USE_MONOT5:
    print("Loading MonoT5 Reranker...")

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

    try:
        mn = "castorini/monot5-base-msmarco-10k"
        monot5_tokenizer = AutoTokenizer.from_pretrained(mn)
        monot5_model = AutoModelForSeq2SeqLM.from_pretrained(mn).to(device)
        monot5_model.eval()
        print(f"  MonoT5 loaded successfully")
    except Exception as e:
        print(f"  MonoT5 load failed: {e}")
        USE_MONOT5 = False

Loading MonoT5 Reranker...


tokenizer_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


pytorch_model.bin:   0%|          | 0.00/892M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

  MonoT5 loaded successfully


## 7. MonoT5 Reranker Function

In [14]:
def rerank_monot5(query, doc_ids, batch_size=16):
    """MonoT5 reranking (pointwise scoring)

    Returns dict of {doc_id: relevance_probability}
    Higher score = more relevant

    batch_size increased to 16 for faster processing
    """
    if not USE_MONOT5 or monot5_model is None:
        return None

    doc_texts, valid_ids = [], []
    for did in doc_ids:
        try:
            doc = index_reader.doc(did)
            if doc:
                raw = doc.raw()
                if raw:
                    doc_texts.append(raw[:2000])
                    valid_ids.append(did)
        except Exception as e:
            pass

    if not doc_texts:
        return None

    prompts = [f"Query: {query} Document: {d} Relevant:" for d in doc_texts]

    true_token_id = monot5_tokenizer.encode("true")[0]
    false_token_id = monot5_tokenizer.encode("false")[0]

    all_scores = []
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i+batch_size]
        try:
            inputs = monot5_tokenizer(
                batch,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=512
            ).to(device)

            with torch.no_grad():
                outputs = monot5_model.generate(
                    **inputs,
                    max_new_tokens=1,
                    return_dict_in_generate=True,
                    output_scores=True
                )
                logits = outputs.scores[0]
                true_logits = logits[:, true_token_id]
                false_logits = logits[:, false_token_id]
                probs = torch.softmax(torch.stack([false_logits, true_logits], dim=1), dim=1)
                batch_scores = probs[:, 1].cpu().tolist()
                all_scores.extend(batch_scores)
        except Exception as e:
            all_scores.extend([0.5] * len(batch))

    return dict(zip(valid_ids, all_scores))

## 10. Phase 2 Pipeline (4-Way Pure RRF with Contriever)

In [15]:
def ultimate_pipeline(query, rerank_depth=1000, k_rrf=10):
    """
    Phase 2: 4-Way Pure RRF with Contriever

    BM25 + RM3 + MonoT5 + Contriever (all weights = 1.0)

    Phase 1 baseline: MAP 0.2855
    Phase 2 target: MAP 0.33-0.36
    """
    # Stage 1: Get BM25, RM3, and Contriever rankings
    bm25_results, rm3_results, contriever_results = multi_method_fusion(query)

    bm25_ranking = {docid: rank for rank, (docid, _) in enumerate(bm25_results, 1)}
    rm3_ranking = {docid: rank for rank, (docid, _) in enumerate(rm3_results, 1)}
    contriever_ranking = {docid: rank for rank, (docid, _) in enumerate(contriever_results, 1)}

    # Stage 2: Get MonoT5 ranking for top docs
    monot5_ranking = {}
    if USE_MONOT5 and monot5_model is not None:
        # Collect top docs from all retrieval methods
        top_docids = set()
        top_docids.update([docid for docid, _ in bm25_results[:rerank_depth]])
        if contriever_results:
            top_docids.update([docid for docid, _ in contriever_results[:rerank_depth]])
        top_docids = list(top_docids)[:rerank_depth]

        monot5_scores = rerank_monot5(query, top_docids)

        if monot5_scores and len(monot5_scores) > 0:
            sorted_by_monot5 = sorted(monot5_scores.items(), key=lambda x: x[1], reverse=True)
            monot5_ranking = {docid: rank for rank, (docid, _) in enumerate(sorted_by_monot5, 1)}

    # Stage 3: 4-Way Pure RRF Fusion (all weights = 1.0)
    all_docids = set(bm25_ranking.keys()) | set(rm3_ranking.keys()) | set(contriever_ranking.keys())
    rrf_scores = {}

    for docid in all_docids:
        rrf_score = 0.0

        # BM25 contribution (weight = 1.0)
        if docid in bm25_ranking:
            rrf_score += 1.0 / (k_rrf + bm25_ranking[docid])

        # RM3 contribution (weight = 1.0)
        if docid in rm3_ranking:
            rrf_score += 1.0 / (k_rrf + rm3_ranking[docid])

        # Contriever contribution (weight = 1.0) - NEW!
        if docid in contriever_ranking:
            rrf_score += 1.0 / (k_rrf + contriever_ranking[docid])

        # MonoT5 contribution (weight = 1.0)
        if docid in monot5_ranking:
            rrf_score += 1.0 / (k_rrf + monot5_ranking[docid])

        rrf_scores[docid] = rrf_score

    # Sort by RRF score (descending)
    final_ranking = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

    # Build results
    results = []
    for rank, (docid, score) in enumerate(final_ranking[:1000], 1):
        results.append((str(docid), float(score), int(rank)))

    return results

## Evaluate on Training Set

In [ ]:
print("="*70)
print("EVALUATING PHASE 2 ON TRAINING SET")
print("="*70)
print("4-Way Pure RRF: BM25 + RM3 + MonoT5 + Contriever")
print(f"Contriever active: {USE_CONTRIEVER}")
print("="*70)
print()

train_results = []
start_time = time.time()

for i, qid in enumerate(train_qids, 1):
    query_text = queries[qid]
    query_type = classify_query(query_text)

    print(f"[{i}/{len(train_qids)}] Query {qid} ({query_type}): {query_text[:50]}...")

    try:
        results = ultimate_pipeline(query_text, rerank_depth=1000)

        for docid, score, rank in results:
            if isinstance(score, np.ndarray):
                score = float(score.item())
            elif isinstance(score, list):
                score = float(score[0]) if score else 0.0
            else:
                score = float(score)

            train_results.append(ir_measures.ScoredDoc(
                query_id=str(qid),
                doc_id=str(docid),
                score=float(score)
            ))

        print(f"  Retrieved {len(results)} docs")
    except Exception as e:
        print(f"  Error: {str(e)[:100]}")
        continue

total_time = time.time() - start_time

# Convert qrels
qrels_list = []
for qid, doc_rels in qrels.items():
    for docid, rel in doc_rels.items():
        qrels_list.append(ir_measures.Qrel(
            query_id=str(qid),
            doc_id=str(docid),
            relevance=int(rel)
        ))

# Calculate metrics
print("\n" + "="*70)
print("COMPUTING METRICS...")
print("="*70)

metrics = [MAP, nDCG@10, nDCG@20, P@10, P@20]
results_metrics = ir_measures.calc_aggregate(metrics, qrels_list, train_results)

print("\n" + "="*70)
print("PHASE 2 RESULTS - TRAINING SET (50 queries)")
print("="*70)
print("\nYour Scores:")
print(f"  MAP:      {results_metrics[MAP]:.4f}  <- Main metric")
print(f"  nDCG@10:  {results_metrics[nDCG@10]:.4f}")
print(f"  nDCG@20:  {results_metrics[nDCG@20]:.4f}")
print(f"  P@10:     {results_metrics[P@10]:.4f}")
print(f"  P@20:     {results_metrics[P@20]:.4f}")

# Compare to Phase 1
phase1_map = 0.2855
current_map = results_metrics[MAP]

print(f"\nComparison:")
print(f"  Phase 1 (3-way): {phase1_map:.4f}")
print(f"  Phase 2 (4-way): {current_map:.4f}")

if current_map >= phase1_map:
    improvement = current_map - phase1_map
    print(f"  Contriever Boost: +{improvement:.4f} (+{improvement/phase1_map*100:.2f}%)")
else:
    decline = phase1_map - current_map
    print(f"  Decline: -{decline:.4f}")

print(f"\nTime: {total_time:.1f}s ({total_time/len(train_qids):.1f}s per query)")

# Assessment
print("\n" + "="*70)
if current_map >= 0.35:
    print("TARGET REACHED! MAP >= 0.35")
elif current_map >= 0.33:
    print("EXCELLENT! Close to target (0.33+)")
elif current_map >= 0.30:
    print("GOOD! Significant improvement")
else:
    print("Check if Contriever loaded correctly")
print("="*70)

EVALUATING PHASE 2 ON TRAINING SET
4-Way Pure RRF: BM25 + RM3 + MonoT5 + Contriever
Contriever active: True

[1/50] Query 301 (short): international organized crime...
  Retrieved 1000 docs
[2/50] Query 302 (short): poliomyelitis post polio...
  Retrieved 1000 docs
[3/50] Query 303 (short): hubble telescope achievements...
  Retrieved 1000 docs
[4/50] Query 304 (short): endangered species mammals...
  Retrieved 1000 docs
[5/50] Query 305 (short): dangerous vehicles...
  Retrieved 1000 docs
[6/50] Query 306 (short): african civilian deaths...
  Retrieved 1000 docs
[7/50] Query 307 (short): new hydroelectric projects...


## Generate Test Results

In [ ]:
print("="*70)
print("GENERATING TEST RESULTS - PHASE 2")
print("="*70)
print(f"Configuration:")
print(f"  4-Way Pure RRF: BM25 + RM3 + MonoT5 + Contriever")
print(f"  Contriever active: {USE_CONTRIEVER}")
print(f"  Rerank depth: 1000 documents")
print(f"  k_rrf: 10")
print(f"  Total test queries: {len(test_qids)}")
print(f"\nExpected: MAP ~{results_metrics[MAP]:.2f} (from training)")
print("="*70)
print()

run3_results = []
start_time = time.time()

for i, qid in enumerate(test_qids, 1):
    query_text = queries[qid]
    query_type = classify_query(query_text)

    print(f"[{i}/{len(test_qids)}] Query {qid} ({query_type}): {query_text[:70]}...")

    query_start = time.time()
    results = ultimate_pipeline(query_text, rerank_depth=1000)
    query_time = time.time() - query_start

    for docid, score, rank in results:
        if isinstance(score, (list, np.ndarray)):
            score = float(score[0]) if len(score) > 0 else 0.0
        else:
            score = float(score)

        run3_results.append({
            'qid': str(qid),
            'docid': str(docid),
            'rank': int(rank),
            'score': float(score)
        })

    print(f"  Retrieved {len(results)} docs in {query_time:.1f}s")

    if i % 20 == 0:
        elapsed = time.time() - start_time
        avg_time = elapsed / i
        remaining = (len(test_qids) - i) * avg_time

        print()
        print("-"*70)
        print(f"PROGRESS: {i}/{len(test_qids)} ({i/len(test_qids)*100:.1f}%)")
        print(f"  Elapsed: {elapsed/60:.1f} min, Remaining: ~{remaining/60:.1f} min")
        print(f"  Avg: {avg_time:.1f}s/query")
        if torch.cuda.is_available():
            print(f"  GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB")
        print("-"*70)
        print()

total_time = time.time() - start_time

print()
print("="*70)
print("GENERATION COMPLETE!")
print("="*70)
print(f"  Queries: {len(test_qids)}")
print(f"  Rankings: {len(run3_results):,}")
print(f"  Total time: {total_time/60:.1f} min")
print(f"  Avg: {total_time/len(test_qids):.1f}s/query")
print("="*70)

## 11. Save Results

In [ ]:
with open('run_3_phase2_contriever.res', 'w') as f:
    for r in run3_results:
        score = r['score']
        if isinstance(score, (list, np.ndarray)):
            score = float(score[0]) if len(score) > 0 else 0.0
        else:
            score = float(score)
        f.write(f"{r['qid']} Q0 {r['docid']} {r['rank']} {score:.6f} run3_phase2_contriever\n")

print("Saved to run_3_phase2_contriever.res")
print("\nFirst 5 lines:")
with open('run_3_phase2_contriever.res', 'r') as f:
    for i, line in enumerate(f):
        if i < 5:
            print(line.strip())
        else:
            break

print("\n" + "="*70)
print("PHASE 2 COMPLETE!")
print("="*70)
print("\nConfiguration:")
print("  BM25: k1=0.7, b=0.6")
print("  RM3: fb_docs=15, fb_terms=100, original_query_weight=0.20")
print("  k_rrf: 10")
print("  rerank_depth: 1000")
print("  4-Way Pure RRF: BM25 + RM3 + MonoT5 + Contriever")
print("\nPerformance:")
print(f"  Phase 1 MAP: 0.2855")
print(f"  Phase 2 MAP: {results_metrics[MAP]:.4f}")
if results_metrics[MAP] >= 0.35:
    print("  TARGET REACHED!")
print("="*70)